# 第14章：推理模型与 GRPO

## 本章目标
- 理解 DeepSeek-R1 的突破——通过纯 RL 训练推理能力
- 掌握 GRPO (Group Relative Policy Optimization) 的原理与实现
- 理解 Process Reward Model (PRM) 与 Outcome Reward Model (ORM) 的区别
- 了解 R1 训练管线与 test-time compute scaling
- 使用 trl GRPOTrainer 完成 GRPO 训练实验

## 前置知识
- 第7章：RLHF (PPO) —— 理解 policy gradient、value network、advantage
- 第8章：DPO —— 理解 reference model、偏好优化

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install torch transformers trl peft datasets accelerate
else:
    print("本地环境运行，请确保已安装: torch, transformers, trl, peft, datasets, accelerate")

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

## 14.1 从 SFT+RLHF 到纯 RL 推理

### DeepSeek-R1-Zero 实验：跳过 SFT，直接用 RL 训练推理

传统 RLHF 流程是 SFT → Reward Model → PPO，但 DeepSeek-R1-Zero 做了一个大胆实验：**跳过 SFT，直接在 base model 上做 RL**。

#### 关键发现

1. **Chain-of-Thought (CoT) 自发涌现**：模型学会在给出答案前展开详细推理过程，而非直接输出结论
2. **Self-correction（自我纠正）**：模型发现推理有误时，会主动回退并修正（"wait, let me reconsider..."）
3. **"Aha moment"**：模型在训练中突然获得新的解题策略，表现出类似"顿悟"的行为

#### 核心洞察

> RL alone can elicit reasoning without human demonstrations.

这意味着：推理能力并不一定需要人类示范来"教"，RL 的奖励信号足以让模型自主发现有效的推理策略。

#### R1-Zero 的问题

虽然 R1-Zero 证明了纯 RL 的可行性，但直接输出存在以下问题：
- 输出格式不统一（推理过程混杂，难以解析）
- 语言混合（中英文混杂）
- 可读性差

因此 DeepSeek-R1（非 Zero 版本）引入了冷启动 SFT + 多阶段管线来解决这些问题（详见 14.4 节）。

参考：[DeepSeek-R1](https://arxiv.org/abs/2501.12948) Section 2

## 14.2 GRPO (Group Relative Policy Optimization)

### 核心创新：消除 Value Network

PPO 需要一个额外的 Value Network (Critic) 来估计 advantage：
$$A(s,a) = r + \gamma V(s') - V(s)$$

但训练 Critic 既昂贵又不稳定。GRPO 的思路是：**用 group baseline 替代 value network**。

### GRPO 算法流程

对于每个 prompt $q$：

1. **采样 G 个输出**：$\{o_1, o_2, ..., o_G\}$ 从当前策略 $\pi_\theta$ 采样
2. **打分**：用 reward model 给每个输出打分 $\{r_1, r_2, ..., r_G\}$
3. **组内归一化 advantage**：
$$\hat{A}_i = \frac{r_i - \text{mean}(r)}{\text{std}(r)}$$
4. **优化 clipped surrogate objective**（类似 PPO）：
$$\mathcal{L}_{GRPO} = \mathbb{E}\left[\min\left(\rho_i \hat{A}_i, \; \text{clip}(\rho_i, 1-\epsilon, 1+\epsilon) \hat{A}_i\right)\right] - \beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$$

其中 $\rho_i = \frac{\pi_\theta(o_i|q)}{\pi_{old}(o_i|q)}$ 是重要性采样比率。

### 与 PPO 的关键区别

| 维度 | PPO | GRPO |
|------|-----|------|
| Value Network | 需要 Critic 网络 | 不需要 |
| Advantage 计算 | $A = r + \gamma V(s') - V(s)$ | 组内 z-score 归一化 |
| Baseline | 由 Critic 学习 | 组内均值 |
| 显存开销 | 高（多一个 Critic） | 低 |
| 适用场景 | 通用 RL | 多采样对比式任务 |

GRPO 的优势在于：同一个 prompt 的多个输出天然形成对比组，
好输出的 advantage 为正（被强化），差输出的 advantage 为负（被抑制）。

参考：[DeepSeek-R1](https://arxiv.org/abs/2501.12948) Section 2.3, [trl GRPOTrainer](https://huggingface.co/docs/trl/main_classes/grpo_trainer)

In [ ]:
def grpo_compute_advantages(rewards: list[float]) -> torch.Tensor:
    """
    计算组内归一化 advantage（GRPO 的核心操作）。

    对于同一 prompt 的 G 个输出，将它们的 reward 做 z-score 归一化。
    - advantage > 0：比组内平均好，应该被强化
    - advantage < 0：比组内平均差，应该被抑制

    Args:
        rewards: 同一 prompt 的 G 个输出的 reward 列表
    Returns:
        advantages: 归一化后的 advantage 张量
    """
    rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
    mean_r = rewards_tensor.mean()
    std_r = rewards_tensor.std()

    # 防止除零：如果所有 reward 相同，advantage 全为 0
    if std_r < 1e-8:
        return torch.zeros_like(rewards_tensor)

    advantages = (rewards_tensor - mean_r) / std_r
    return advantages


# 演示：模拟一个 prompt 的 4 个输出的 reward
rewards_example = [0.8, 0.3, 1.0, 0.1]
advantages = grpo_compute_advantages(rewards_example)

print("=== GRPO Advantage 计算示例 ===")
print(f"Rewards:    {rewards_example}")
print(f"Mean:       {torch.tensor(rewards_example).mean():.4f}")
print(f"Std:        {torch.tensor(rewards_example).std():.4f}")
print(f"Advantages: {[f'{a:.4f}' for a in advantages.tolist()]}")
print()
print("解读：")
print(f"  输出 0 (reward=0.8): advantage={advantages[0]:.4f} > 0 → 比平均好，被强化")
print(f"  输出 1 (reward=0.3): advantage={advantages[1]:.4f} < 0 → 比平均差，被抑制")
print(f"  输出 2 (reward=1.0): advantage={advantages[2]:.4f} > 0 → 最好，被强化最多")
print(f"  输出 3 (reward=0.1): advantage={advantages[3]:.4f} < 0 → 最差，被抑制最多")

In [ ]:
def grpo_loss(log_probs_new: torch.Tensor,
              log_probs_old: torch.Tensor,
              log_probs_ref: torch.Tensor,
              advantages: torch.Tensor,
              clip_epsilon: float = 0.2,
              kl_coef: float = 0.01) -> torch.Tensor:
    """
    手写简化版 GRPO loss（约 50 行核心逻辑）。

    Args:
        log_probs_new:  当前策略 π_θ 对每个输出的 log probability, shape [G]
        log_probs_old:  旧策略 π_old 对每个输出的 log probability, shape [G]
        log_probs_ref:  参考策略 π_ref 对每个输出的 log probability, shape [G]
        advantages:     组内归一化的 advantage, shape [G]
        clip_epsilon:   PPO 裁剪参数
        kl_coef:        KL 惩罚系数 β
    Returns:
        loss: 标量 loss（要最小化，因此取负号）
    """
    # 1. 计算重要性采样比率 ratio = π_new / π_old
    log_ratio = log_probs_new - log_probs_old
    ratio = torch.exp(log_ratio)  # shape [G]

    # 2. Clipped surrogate objective（PPO clip）
    #    对 advantage > 0 的输出：ratio 被限制在 (1+ε)，防止过度强化
    #    对 advantage < 0 的输出：ratio 被限制在 (1-ε)，防止过度抑制
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()  # 取负号因为要做梯度下降

    # 3. KL 散度惩罚：防止策略偏离 reference 太远
    #    D_KL(π_θ || π_ref) ≈ E[log(π_θ / π_ref)]
    kl_penalty = kl_coef * (log_probs_new - log_probs_ref).mean().abs()

    # 4. 总 loss
    loss = policy_loss + kl_penalty
    return loss


# ===== 演示：模拟 GRPO 的一个训练步 =====
G = 4  # 每个 prompt 生成 4 个输出
torch.manual_seed(42)

# 模拟 log probabilities（实际中由模型 forward 计算得到）
log_probs_old = torch.tensor([-2.1, -3.5, -1.8, -4.2])  # 旧策略
log_probs_new = torch.tensor([-1.9, -3.3, -1.6, -4.5])  # 当前策略（轻微更新）
log_probs_ref = torch.tensor([-2.0, -3.4, -1.7, -4.3])  # 参考策略

# 模拟 reward 并计算 advantage
rewards = [0.8, 0.2, 1.0, 0.1]
advantages = grpo_compute_advantages(rewards)

# 计算 GRPO loss
loss = grpo_loss(log_probs_new, log_probs_old, log_probs_ref, advantages)

print("=== GRPO Loss 计算示例 ===")
print(f"Outputs:    G={G} 个输出")
print(f"Rewards:    {rewards}")
print(f"Advantages: {[f'{a:.4f}' for a in advantages.tolist()]}")
print(f"GRPO Loss:  {loss.item():.4f}")
print()
print("关键理解：")
print(f"  - 输出 2 (best, r=1.0): advantage={advantages[2]:.4f}, ratio={torch.exp(log_probs_new[2] - log_probs_old[2]):.4f}")
print(f"  - 输出 3 (worst, r=0.1): advantage={advantages[3]:.4f}, ratio={torch.exp(log_probs_new[3] - log_probs_old[3]):.4f}")
print(f"  - 梯度会让好的输出 (advantage>0) 的概率更高，差的输出 (advantage<0) 的概率更低")

### GRPO 直觉理解

GRPO 的核心思想可以类比为"相对排名"：

1. **同一个问题，多个人回答**：模型对同一个 prompt 生成 G 个不同的回答
2. **互相比较**：不依赖绝对分数，而是看每个回答相对于组内平均的表现
3. **奖优罚劣**：比平均好的被强化，比平均差的被抑制
4. **clip 保险**：防止某一次更新幅度过大（与 PPO 相同的安全机制）

之所以不需要 Value Network，是因为**组内多个样本的 reward 分布本身就提供了 baseline**。
这类似于让同一批学生做同一道题，用班级平均分作为基准来评价每个学生的表现。

GRPO 特别适合推理任务，因为：
- 推理任务的 reward 信号明确（答案对/错、部分分数）
- 多次采样的成本可控（推理题通常较短）
- 组内对比能准确区分"好的推理路径"和"差的推理路径"

---

## 14.3 Process Reward Model (PRM)

### Outcome RM vs Process RM

**Outcome Reward Model (ORM)**：只对最终答案打一个分数
- 优点：简单，标注成本低
- 缺点：奖励信号稀疏——一个多步推理只有结尾有一个分数

**Process Reward Model (PRM)**：对推理的**每一步**打分
- 优点：奖励信号更密集，能定位哪一步出了错
- 缺点：标注成本高（需要人类标注每一步的正确性）

### 直觉对比

```
问题: 求 (3 + 5) × 2 的值

推理过程:
  Step 1: 3 + 5 = 8       ← PRM: 1.0 (正确)
  Step 2: 8 × 2 = 16      ← PRM: 1.0 (正确)
  Answer: 16               ← ORM: 1.0 (正确)

对比错误案例:
  Step 1: 3 + 5 = 8       ← PRM: 1.0 (正确)
  Step 2: 8 × 2 = 14      ← PRM: 0.0 (错误！)
  Answer: 14               ← ORM: 0.0 (最终答案错)
```

ORM 只知道最终答案错了，但不知道哪一步出错。
PRM 能精确定位 Step 2 的计算错误，提供更有信息量的学习信号。

### PRM 对推理训练的意义

对于多步推理任务（数学、逻辑、代码），PRM 的优势尤为明显：
- **更密集的反馈**：每一步都有 reward，而非只有最终结果
- **更精确的 credit assignment**：能区分"第一步就错了"和"最后一步才错"
- **更好的搜索引导**：在 tree search 中，PRM 可以剪枝掉早期就出错的路径

参考：[Let's Verify Step by Step](https://arxiv.org/abs/2305.20050) (Lightman et al., 2023)

In [ ]:
# 演示 ORM vs PRM 的奖励差异

class MockReasoningTrace:
    """模拟一个多步推理过程"""
    def __init__(self, steps: list[str], step_correct: list[bool]):
        self.steps = steps
        self.step_correct = step_correct
        self.final_correct = step_correct[-1] if step_correct else False

    def orm_score(self) -> float:
        """Outcome RM: 只看最终答案是否正确"""
        return 1.0 if self.final_correct else 0.0

    def prm_scores(self) -> list[float]:
        """Process RM: 对每一步打分"""
        return [1.0 if c else 0.0 for c in self.step_correct]

    def prm_total(self) -> float:
        """PRM 总分：所有步骤分数之和"""
        return sum(self.prm_scores())


# 案例 1：推理完全正确
trace_good = MockReasoningTrace(
    steps=["3 + 5 = 8", "8 × 2 = 16"],
    step_correct=[True, True]
)

# 案例 2：第一步对，第二步错（中间出错）
trace_mid = MockReasoningTrace(
    steps=["3 + 5 = 8", "8 × 2 = 14"],
    step_correct=[True, False]
)

# 案例 3：第一步就错
trace_bad = MockReasoningTrace(
    steps=["3 + 5 = 7", "7 × 2 = 14"],
    step_correct=[False, False]
)

print("=== ORM vs PRM 对比 ===")
print(f"{'案例':<10} {'ORM':<8} {'PRM (逐步)':<20} {'PRM 总分':<10}")
print("-" * 50)
for name, trace in [("完全正确", trace_good), ("中间出错", trace_mid), ("第一步错", trace_bad)]:
    print(f"{name:<10} {trace.orm_score():<8.1f} {str(trace.prm_scores()):<20} {trace.prm_total():<10.1f}")

print()
print("关键观察：")
print(f"  ORM 下：'中间出错'和'第一步错'得分相同 (都是 0.0)，无法区分")
print(f"  PRM 下：'中间出错'得分 (1.0) > '第一步错' (0.0)，能区分错误位置")
print(f"  PRM 提供了更密集、更有信息量的奖励信号")

---

## 14.4 R1 训练管线

DeepSeek-R1 采用了多阶段训练管线，而非 R1-Zero 的纯 RL 方案：

```
┌─────────────────────────────────────────────────────────────────┐
│                    DeepSeek-R1 训练管线                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Stage 1: Cold-start SFT（冷启动微调）                          │
│  ┌──────────────────────────────────────────┐                   │
│  │ Base Model → SFT with few long-CoT data  │                   │
│  │ 目的：统一输出格式，教会 <think</think</think                  │
│  └──────────────────────┬───────────────────┘                   │
│                         ▼                                       │
│  Stage 2: RL with GRPO（推理能力 RL 训练）                      │
│  ┌──────────────────────────────────────────┐                   │
│  │ GRPO on math/code/science tasks          │                   │
│  │ Reward: correctness + format + language   │                   │
│  └──────────────────────┬───────────────────┘                   │
│                         ▼                                       │
│  Stage 3: Rejection Sampling + SFT（拒绝采样 + SFT）            │
│  ┌──────────────────────────────────────────┐                   │
│  │ 从 RL 模型采样，筛选高质量 CoT 数据       │                   │
│  │ 合并通用 SFT 数据，重新做 SFT              │                   │
│  └──────────────────────┬───────────────────┘                   │
│                         ▼                                       │
│  Stage 4: All-task RL（全场景 RL）                              │
│  ┌──────────────────────────────────────────┐                   │
│  │ 在所有场景上做 RL（推理+写作+对话+...）    │                   │
│  │ 不同场景使用不同 reward signal             │                   │
│  └──────────────────────┬───────────────────┘                   │
│                         ▼                                       │
│  Stage 5: Distillation（蒸馏）                                  │
│  ┌──────────────────────────────────────────┐                   │
│  │ R1 (671B) → 蒸馏到更小的模型 (1.5B~70B)  │                   │
│  │ 蒸馏效果 > 直接在小模型上做 RL             │                   │
│  └──────────────────────────────────────────┘                   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 各阶段要点

**Stage 1 - 冷启动 SFT**
- 用少量高质量的 long-CoT 数据做 SFT
- 目的是让模型学会结构化输出（如 `<think</think` 标签包裹推理过程）
- 解决 R1-Zero 的输出格式混乱问题

**Stage 2 - RL with GRPO**
- 在推理密集型任务（数学、代码、科学）上做 GRPO
- 使用 correctness reward（答案是否正确）+ format reward（格式是否规范）
- 这是最关键的推理能力提升阶段

**Stage 3 - Rejection Sampling + SFT**
- 从 Stage 2 的模型大量采样，筛选高质量推理数据
- 与通用 SFT 数据合并，重新微调
- 目的：让模型既保持推理能力，又不丧失通用能力

**Stage 4 - 全场景 RL**
- 在所有任务场景上做 RL
- 不同场景使用不同的 reward：推理用 correctness，写作用 helpfulness，安全用 safety

**Stage 5 - 蒸馏**
- 将大模型的推理能力迁移到小模型
- DeepSeek 发现：蒸馏的效果 > 在小模型上直接做 RL

参考：[DeepSeek-R1](https://arxiv.org/abs/2501.12948) Section 3

---

## 14.5 Test-time Compute Scaling

### 推理阶段的计算量也能提升效果

传统观点认为模型能力的提升主要靠训练时的计算量（train-time compute）。
但最新研究表明，**在推理阶段投入更多计算同样可以显著提升效果**。

### 主要方法

**1. Majority Voting（多数投票）**
- 让模型对同一问题生成 N 个回答
- 取出现次数最多的答案作为最终答案
- 简单有效，但需要 N 次推理

**2. Best-of-N**
- 生成 N 个回答，用 verifier/reward model 选最好的
- 效果比 majority voting 更好（有判断标准）

**3. Tree Search（树搜索）**
- 在推理过程中构建搜索树，每一步探索多个候选
- 用 PRM 引导搜索方向（每步的得分作为启发式）
- 计算成本最高，但效果也最好

**4. Verification（验证）**
- 模型生成答案后，让另一个模型（或自身）验证答案的正确性
- 结合 self-correction 实现迭代优化

### Scaling 分析

```
Accuracy
  ↑
  │           ╱ Tree Search + PRM
  │         ╱
  │       ╱ Best-of-N + Verifier
  │     ╱
  │   ╱ Majority Voting
  │ ╱
  │╱ Single Pass
  └───────────────────→ Inference Compute
```

关键发现：在相同的总计算预算下，用更小的模型 + 更多推理计算，
有时可以匹敌甚至超越更大模型的 single-pass 表现。

参考：[Scaling LLM Test-Time Compute](https://arxiv.org/abs/2408.03314) (Snell et al., 2024)

In [ ]:
# 演示 majority voting 和 best-of-N 的效果

import random
from collections import Counter

random.seed(42)

# 模拟一个模型在数学题上的表现
# 假设模型答对的概率是 0.6
model_accuracy = 0.6

def simulate_answer(correct_prob: float) -> str:
    """模拟模型的一次回答"""
    return "correct" if random.random() < correct_prob else "wrong"

def simulate_with_score(correct_prob: float) -> tuple[str, float]:
    """模拟一次回答和对应的 reward model 分数"""
    is_correct = random.random() < correct_prob
    answer = "correct" if is_correct else "wrong"
    # 正确答案倾向于高分，但有噪声
    score = random.gauss(0.8 if is_correct else 0.3, 0.15)
    return answer, score

def majority_vote(correct_prob: float, n: int) -> bool:
    """Majority voting: 生成 N 个回答，取多数"""
    answers = [simulate_answer(correct_prob) for _ in range(n)]
    most_common = Counter(answers).most_common(1)[0][0]
    return most_common == "correct"

def best_of_n(correct_prob: float, n: int) -> bool:
    """Best-of-N: 生成 N 个回答，用 reward model 选最高分的"""
    results = [simulate_with_score(correct_prob) for _ in range(n)]
    best = max(results, key=lambda x: x[1])
    return best[0] == "correct"

# 跑 1000 次实验，统计各方法准确率
num_trials = 1000
N_values = [1, 5, 10, 20, 50]

print("=== Test-time Compute Scaling 效果演示 ===")
print(f"模型单次准确率: {model_accuracy:.0%}")
print(f"每个配置跑 {num_trials} 次实验")
print()
print(f"{'N':>5} | {'Single Pass':>12} | {'Majority Vote':>14} | {'Best-of-N':>10}")
print("-" * 50)

for n in N_values:
    single = sum(1 for _ in range(num_trials) if simulate_answer(model_accuracy) == "correct") / num_trials
    mv_acc = sum(1 for _ in range(num_trials) if majority_vote(model_accuracy, n)) / num_trials
    bon_acc = sum(1 for _ in range(num_trials) if best_of_n(model_accuracy, n)) / num_trials
    print(f"{n:>5} | {single:>12.1%} | {mv_acc:>14.1%} | {bon_acc:>10.1%}")

print()
print("观察：")
print("  - 随着 N 增大，Majority Vote 和 Best-of-N 的准确率都显著提升")
print("  - Best-of-N (有 reward model) 效果优于 Majority Vote")
print("  - 代价是推理计算量线性增长（需要 N 次推理）")

---

## 14.6 动手实验：使用 GRPOTrainer 训练数学推理

本节使用 trl 的 GRPOTrainer，在 Qwen2.5-0.5B 上用 LoRA 做 GRPO 训练。

任务：GSM8K 小学数学推理。

奖励函数：检查模型输出的最终答案是否正确（Outcome Reward）。

注意：此实验需要 GPU（推荐 Colab T4 或更好）。如果无 GPU，代码会给出说明。

In [ ]:
# 检查 GPU 可用性
import torch

CUDA_AVAILABLE = torch.cuda.is_available()

if CUDA_AVAILABLE:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU 可用: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠ GPU 不可用")
    print("本实验需要 GPU。你可以：")
    print("  1. 在 Google Colab 上运行（选择 T4 GPU runtime）")
    print("  2. 跳过训练部分，仅阅读代码和理论")
    print()
    print("以下代码会加载模型但跳过实际训练。")
    print("建议在 Colab 上完成完整实验。")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType
from datasets import load_dataset
import re

# 模型选择：Qwen2.5-0.5B（足够小，适合 T4 显存）
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"加载模型: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

if CUDA_AVAILABLE:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    print(f"模型加载完成，参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
else:
    print("无 GPU，跳过模型加载。请在 Colab 环境中运行。")
    model = None

In [ ]:
# 加载 GSM8K 数据集
# GSM8K 是小学数学题数据集，每题包含 question 和 answer

if CUDA_AVAILABLE:
    dataset = load_dataset("openai/gsm8k", "main", split="train[:500]")
    print(f"数据集大小: {len(dataset)}")
    print(f"样例问题: {dataset[0]['question'][:100]}...")
    print(f"样例答案: {dataset[0]['answer'][:100]}...")
else:
    print("无 GPU，跳过数据集加载。")
    dataset = None

# GSM8K answer 格式: "推理过程 #### 最终数字答案"
# 我们需要提取最终数字答案用于 reward 计算
def extract_answer(answer_text: str) -> str:
    """从 GSM8K 格式中提取最终数字答案"""
    if "####" in answer_text:
        return answer_text.split("####")[-1].strip()
    return answer_text.strip()

# 测试提取函数
sample_answer = "Janet's ducks lay 16 eggs per day. She eats three. So 16 - 3 = 13. #### 13"
print(f"\n提取测试: '{extract_answer(sample_answer)}'")

In [ ]:
# 定义 Reward Function
# GRPOTrainer 需要一个 reward function，输入 prompts 和 completions，输出 reward 列表

def math_reward_function(prompts: list[str], completions: list[str],
                         ground_truths: list[str] = None) -> list[float]:
    """
    数学推理的 reward function。
    检查模型输出的最终答案是否与标准答案匹配。

    Args:
        prompts: 输入的问题列表
        completions: 模型生成的回答列表
        ground_truths: 标准答案列表
    Returns:
        rewards: 每个 completion 的 reward（0.0 或 1.0）
    """
    rewards = []
    for completion, gt in zip(completions, ground_truths):
        # 尝试从模型输出中提取数字答案
        # 常见模式："答案是 X"、"answer is X"、"#### X"、最后出现的数字
        numbers = re.findall(r'-?\d+\.?\d*', completion)
        if numbers:
            model_answer = numbers[-1].strip()  # 取最后一个数字
            gt_clean = re.findall(r'-?\d+\.?\d*', gt)
            if gt_clean:
                gt_answer = gt_clean[-1].strip()
                if model_answer == gt_answer:
                    rewards.append(1.0)
                else:
                    rewards.append(0.0)
            else:
                rewards.append(0.0)
        else:
            rewards.append(0.0)  # 没有输出数字，给 0 分
    return rewards


# 测试 reward function
test_prompts = ["What is 2 + 3?"]
test_completions = ["Let me calculate. 2 + 3 = 5. The answer is 5."]
test_ground_truths = ["5"]

reward = math_reward_function(test_prompts, test_completions, test_ground_truths)
print(f"测试 reward function:")
print(f"  问题: {test_prompts[0]}")
print(f"  回答: {test_completions[0]}")
print(f"  Reward: {reward[0]}")

# 错误回答测试
test_wrong = ["2 + 3 = 6. The answer is 6."]
reward_wrong = math_reward_function(test_prompts, test_wrong, test_ground_truths)
print(f"\n错误回答: {test_wrong[0]}")
print(f"  Reward: {reward_wrong[0]}")

In [ ]:
# 配置并运行 GRPOTrainer

if CUDA_AVAILABLE and model is not None and dataset is not None:
    from trl import GRPOTrainer, GRPOConfig

    # LoRA 配置：只训练少量参数，节省显存
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
    )

    # GRPO 训练配置
    grpo_config = GRPOConfig(
        output_dir="./grpo_output",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        learning_rate=5e-5,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        max_completion_length=256,   # 最大生成长度
        num_generations=4,           # 每个 prompt 生成 4 个候选（G=4）
        temperature=0.7,             # 采样温度
        bf16=False,                  # T4 不支持 bf16，用 fp16
        fp16=True,
    )

    # 准备数据集格式
    def format_gsm8k(example):
        return {
            "prompt": example["question"],
            "ground_truth": extract_answer(example["answer"]),
        }

    train_dataset = dataset.map(format_gsm8k)

    # GRPOTrainer 需要的 reward function 签名
    def reward_fn(completions, **kwargs):
        prompts = kwargs.get("prompt", [""] * len(completions))
        ground_truths = kwargs.get("ground_truth", [""] * len(completions))
        return math_reward_function(prompts, completions, ground_truths)

    trainer = GRPOTrainer(
        model=model,
        args=grpo_config,
        train_dataset=train_dataset,
        processing_class=tokenizer,
        peft_config=lora_config,
        reward_funcs=reward_fn,
    )

    print("GRPOTrainer 初始化完成")
    print(f"配置: num_generations={grpo_config.num_generations}, "
          f"max_completion_length={grpo_config.max_completion_length}")
    print(f"数据集: {len(train_dataset)} 样本")
    print("\n开始训练（这可能需要 10-30 分钟，取决于 GPU）...")

    trainer.train()
    print("\nGRPO 训练完成！")

else:
    print("=" * 60)
    print("无 GPU，跳过 GRPOTrainer 训练。")
    print()
    print("完整的 GRPO 训练流程如下：")
    print("1. 加载 Qwen2.5-0.5B-Instruct")
    print("2. 配置 LoRA (r=16, target: q_proj, v_proj)")
    print("3. 配置 GRPOConfig (num_generations=4, temperature=0.7)")
    print("4. 定义 reward function (数学答案正确性检查)")
    print("5. 创建 GRPOTrainer 并训练")
    print()
    print("在 Colab T4 上预计训练时间：约 10-30 分钟 (500 样本, 1 epoch)")
    print("=" * 60)

In [ ]:
# 训练后测试：让模型做几道数学题

if CUDA_AVAILABLE and model is not None:
    test_questions = [
        "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends on Sunday. How many eggs does she have left at the end of the week?",
        "A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?",
        "Josh decides to try flipping a house. He buys a house for $80,000 and then puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?",
    ]

    print("=== GRPO 训练后模型测试 ===")
    print()

    for i, q in enumerate(test_questions):
        inputs = tokenizer(q, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
            )
        response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"问题 {i+1}: {q[:80]}...")
        print(f"回答: {response[:200]}")
        print("-" * 60)

else:
    print("无 GPU，跳过训练后测试。")
    print("在完整训练后，模型应该展现出：")
    print("  1. 更结构化的推理过程（分步计算）")
    print("  2. 更高的数学题准确率")
    print("  3. 自我检查和纠正行为（可选）")

### GRPO 实验分析

在完成 GRPO 训练后，应关注以下指标：

- **Reward 趋势**：训练过程中 reward 应逐步上升，说明模型在学到更好的推理策略
- **Completion Length**：推理过程可能变长（更多 CoT），这是正常现象
- **准确率**：在 GSM8K 测试集上的准确率应有所提升

与第7章的 PPO 训练对比：
- GRPO 不需要 Critic 网络，显存开销更低
- GRPO 的 group baseline 比 learned baseline 更稳定（不会出现 Critic 训不好的情况）
- GRPO 特别适合 reward 信号明确的任务（如数学题的答案对/错）

---

## 本章总结

1. **DeepSeek-R1-Zero** 证明了纯 RL（不需要 SFT）可以激发模型的推理能力
2. **GRPO** 用 group baseline 替代 value network，简化了 RL 训练
3. **PRM** 提供逐步奖励信号，比 ORM 更适合多步推理训练
4. **R1 管线** 通过冷启动 SFT + GRPO + 拒绝采样 + 蒸馏实现完整训练
5. **Test-time compute** 表明推理阶段的计算投入也能显著提升效果

### 推理模型的核心范式转变

传统：SFT 教会模型"怎么推理" → RLHF 微调偏好

新范式：RL 让模型"自己发现怎么推理" → 蒸馏传播推理能力

这个范式转变的影响深远：
- 降低了推理数据的标注成本（不需要标注推理过程）
- 模型能发现人类未曾想到的推理策略
- 推理能力可以通过蒸馏高效迁移到小模型

## 参考文献

- [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948)
- [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050) (Lightman et al., 2023)
- [Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters](https://arxiv.org/abs/2408.03314) (Snell et al., 2024)
- [trl GRPOTrainer 文档](https://huggingface.co/docs/trl/main_classes/grpo_trainer)

## 练习

1. **调整 GRPO 超参**：修改 `num_generations` (2, 4, 8) 和 `temperature` (0.5, 0.7, 1.0)，观察训练效果的变化
2. **对比 PPO vs GRPO**：在第7章的 PPO 实验和本章的 GRPO 实验中，比较显存使用、训练稳定性和最终效果
3. **设计 PRM reward**：修改 reward function，不仅检查最终答案，还检查中间推理步骤（提示：用正则匹配中间计算结果）
4. **Test-time scaling 实验**：对训练后的模型分别用 majority voting (N=1,5,10) 和 best-of-N (N=1,5,10) 评估，绘制 scaling 曲线
5. **思考题**：GRPO 的 group baseline 在什么情况下会失效？（提示：当所有输出的 reward 相同时会发生什么？）